# Predict Podcast Listening Time
Kaggle Competition Link: https://www.kaggle.com/competitions/playground-series-s5e4/overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

## Data Loading

In [ ]:
# Loading Training and Testing Datasets and Sample Submission File from links
df =  pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/train.csv")
df_test = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/test.csv")
df_sample = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/sample_submission.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check unique element for each column
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

## Data Cleaning

In [ ]:
# Check rows where Episode_Length_minutes > 300
df_test[df_test["Episode_Length_minutes"] > 300]

In [ ]:
# Change the value for rows with Episode_Length_minutes with the mean of Episode_Length_minutes based on the Podcast Name excluding such rows
df_test.loc[df_test["Episode_Length_minutes"] > 300, "Episode_Length_minutes"] = df_test.loc[df_test["Episode_Length_minutes"] > 300, "Podcast_Name"].map(df.groupby("Podcast_Name")["Episode_Length_minutes"].mean())

In [ ]:
df_test.iloc[[54434, 56597]]

In [ ]:
# Create column that distinguishes train data with '1' or '0' for test. Combine datasets after
df["is_train"] = 1
df_test["is_train"] = 0
df_combined = pd.concat([df, df_test])

In [ ]:
df_combined.head()

In [ ]:
df_combined.describe()

In [ ]:
# Fill null values for Episode_Length_minutes, Guest_Popularity_percentage, and Number_of_Ads based on Podcast Name
df_combined["Episode_Length_minutes"] = df_combined.groupby("Podcast_Name")["Episode_Length_minutes"].transform(lambda x: x.fillna(x.mean()))
df_combined["Guest_Popularity_percentage"] = df_combined.groupby("Podcast_Name")["Guest_Popularity_percentage"].transform(lambda x: x.fillna(x.mean()))
df_combined["Number_of_Ads"] = df_combined.groupby("Podcast_Name")["Number_of_Ads"].transform(lambda x: x.fillna(x.mean()))

In [ ]:
df_combined.describe()

In [ ]:
df_combined.duplicated().sum()

In [ ]:
df_combined.info()

In [ ]:
numerical_cols = df_combined.select_dtypes(include=[np.number]).columns.tolist()
# Remove is_train, id
numerical_cols.remove("is_train")
numerical_cols.remove("id")
numerical_cols.remove("Listening_Time_minutes")

In [ ]:
numerical_cols

## Exploratory Data Analysis

In [ ]:
# Box Plot of listening time based on Genre
fig = plt.box(df_combined, x="Genre", y="Listening_Time_minutes", title="Listening Time by Genre")
fig.show()

In [ ]:
# Box Plot of listening time based on Episode_sentiment
fig = px.box(df_combined, x="Episode_Sentiment", y="Listening_Time_minutes", title="Listening Time by Episode_Sentiment")
fig.show()

In [ ]:
# Box Plot of Listening Time based on Publication_Day
fig = px.box(df_combined, x="Publication_Day", y="Listening_Time_minutes", title="Listening Time by Publication_Day")
fig.show()

In [ ]:
# Box Plot Listening Time Based on Publication_Time
fig = px.box(df_combined, x="Publication_Time", y="Listening_Time_minutes", title="Listening Time by Publication_Time")
fig.show()

In [ ]:
# Box Plot Listening Time based on Episode_Title
fig = px.box(df_combined, x="Episode_Title", y="Listening_Time_minutes", title="Listening Time by Episode_Title")
fig.show()

In [ ]:
# Box Plot Listening Time based on Podcast_Name
fig = px.box(df_combined, x="Podcast_Name", y="Listening_Time_minutes", title="Listening Time by Podcast_Name")
fig.show()

In [ ]:
# Scatter plot of listening time against Host_popularity_percentage
fig = px.scatter(df_combined, x="Host_Popularity_percentage", y="Listening_Time_minutes", title="Listening Time vs Host Popularity")
fig.show()

In [ ]:
# Scatter plot of listening time against Guest_popularity_percentage
fig = px.scatter(df_combined, x="Guest_Popularity_percentage", y="Listening_Time_minutes", title="Listening Time vs Guest Popularity")
fig.show()

## Feature Engineering

In [ ]:
# Scale Numerical Columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_combined[numerical_cols] = scaler.fit_transform(df_combined[numerical_cols])

**Label Encoding Treatments**
- Nominal Encoding: Podcast_Name, Genre
- Ordinal Encoding: Episode_Title, Publication_Day, Publication _Time, Episode_Sentiment

In [ ]:
# Label Encoder for Nominal Encoding
le = LabelEncoder()
df_combined["Podcast_Name"] = le.fit_transform(df_combined["Podcast_Name"])
df_combined["Genre"] = le.fit_transform(df_combined["Genre"])

In [ ]:
# Remove Episode string and convert string to int
df_combined["Episode_Title"] = df_combined["Episode_Title"].str.replace("Episode", "").astype(int)

In [ ]:
# Check unique values for Publication_Day, Publication_time, and Episode_Sentiment
print(df_combined["Publication_Day"].unique())
print(df_combined["Publication_Time"].unique())
print(df_combined["Episode_Sentiment"].unique())

In [ ]:
# Set Monday as 1, Tuesday as 2, and so on
df_combined["Publication_Day"] = df_combined["Publication_Day"].map({"Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4, "Friday": 5, "Saturday": 6, "Sunday": 7})

In [ ]:
# Set Morning as 1, Afternoon as 2, Evening as 3, and Night as 4
df_combined["Publication_Time"] = df_combined["Publication_Time"].map({"Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4})

In [ ]:
# Set Negative as 0, Neutral as 1, and Positive as 2
df_combined["Episode_Sentiment"] = df_combined["Episode_Sentiment"].map({"Negative": 0, "Neutral": 1, "Positive": 2})

In [ ]:
# Separate dataset into df and df_test based on df_train
df = df_combined[df_combined["is_train"] == 1]
df_test = df_combined[df_combined["is_train"] == 0]

## Training and Test Splits

In [ ]:
X = df.drop(["Listening_Time_minutes", "is_train", "id"], axis=1)
y = df["Listening_Time_minutes"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training

In [ ]:
# Implement LightGBM, XGBoost, and GBM as Model. Use Default Parameters to serve as baseline
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
model = LGBMRegressor()
model.fit(X_train, y_train)

## Model Evaluation

In [ ]:
# Check for Root Mean Square error and R2 Score
y_pred = model.predict(X_test)
print(f"Root Mean Square Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")

## Preparing Submission File

In [ ]:
id = df_sample.pop('id')
df_test = df_test.drop(["Listening_Time_minutes", "is_train", "id"], axis=1)
y_pred = model.predict(df_test)

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'id': id,
    'Listening_Time_minutes': y_pred
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")